# Workshop: Flood Arrival Times Estimation with Convolutional Neural Networks
Many countries such as The Netherlands are subjected to flood risk.
To protect from this hazard, low-lying areas are often proteced by longitudinal wall-like strucutures, such as dikes (or levees).

However, these flood defences can be affected by local failures, due to excessive hydraulic loads (e.g., the flood water overtops the dike) or material weaknesses, among the many reasons.
This leads to so-called dike breach floods, whose main danger is their unexpectedness.
Flood practitioners thus require to understand where and when flood water will reach a given location in the area at risk in order to set up evacuation plans.
A common way to analyse this is via flood arrival time (FAT) maps, which indicate, for each point in the domain, how long it takes for the flood to reach it.


```{figure} ../../figures/copyrighted/flood_arrival_times.png
:scale: 45%
:name: flood_arrival_times

Example of flood arrival time (FAT) map, taken from "Ferrari, A., Dazzi, S., Vacondio, R. and Mignosa, P., 2020. Enhancing the resilience to flooding induced by levee breaches in lowland areas: a methodology based on numerical modelling. Natural Hazards and Earth System Sciences, 20(1), pp.59-72."
```

## Objective

Numerical models for determining the flood arrival times can be very slow, depending on the domain size and complexity.
Moreover, to consider uncertainties in breach formation and location, many simulations must be run, resulting in excessive computational constraints.

Your objective is develop a surrogate model, i.e., a model that replicates the output of another model but faster.
For this exercise, you will use a dataset of pre-run simulations in which the topography changes across the different samples.
You will develop a U-NET style CNN that takes topographical information given by elevation and slopes and predicts the flood arrival times.

```{figure} ../../figures/copyrighted/unet.png
:scale: 45%
:name: unet

Example of U-NET architecture, taken from "Ronneberger, O., Fischer, P. and Brox, T., 2015. U-net: Convolutional networks for biomedical image segmentation. In Medical Image Computing and Computer-Assisted Intervention–MICCAI 2015: 18th International Conference, Munich, Germany, October 5-9, 2015, Proceedings, Part III 18 (pp. 234-241). Springer International Publishing."
```

## Libraries

In [1]:
## Useful libraries
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import os
import copy
import pickle
from urllib.request import urlretrieve
from torch.utils.data import DataLoader
from torch.utils.data.dataset import random_split
from sklearn.preprocessing import MinMaxScaler
from matplotlib.colors import TwoSlopeNorm

from cycler import cycler
import seaborn as sns

# Set the color scheme
sns.set_theme()
colors = ['#0076C2', '#EC6842', '#A50034', '#009B77', '#FFB81C', '#E03C31', '#6CC24A', '#EF60A3', '#0C2340', '#00B8C8', '#6F1D77']
plt.rcParams['axes.prop_cycle'] = cycler(color=colors)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
print(device)

## Dataset

First load the dataset and explore what the training samples look like.

In [ ]:
train_file = "train_dataset.pk"

# Download the flood arrival time dataset (if necessary)
train_url = "https://surfdrive.surf.nl/s/TdFC5qbgKxz7xSi/download"
test_url = "https://surfdrive.surf.nl/s/pNnq4iHcNxDTRdD/download"

if not os.path.isfile(train_file):
    print("Downloading FAT dataset...")
    urlretrieve(train_url, "train_dataset.pk")
    urlretrieve(test_url, "test_dataset.pk")

with open('train_dataset.pk', 'rb') as f:
    train_dataset = pickle.load(f)

with open('test_dataset.pk', 'rb') as f:
    test_dataset = pickle.load(f)

### Plot the inputs and outputs for one example.

Each example corresponds to one full flood simulation, carried out assuming a dike breach flood with constant discharge of $50m^3/s$, starting in the bottom-left corner of the domain.
The domain is a 64x64 grid where each tile/patch is $100m$ in length.

Among the simulations, we change the spatial distribution of the topography.

The inputs and outputs are as follows:

Inputs:
- digital elevation model (DEM)
- slope in the x direction
- slope in the y direction

Output:
- flood arrival time (FAT) map

In [ ]:
inputs, outputs = train_dataset[0]

fig, axs = plt.subplots(1, 4, figsize=(10, 5))

axs[0].imshow(inputs[0].cpu(), cmap='terrain', origin='lower')
axs[0].set_title('DEM')

axs[1].imshow(inputs[1].cpu(), cmap='RdBu', origin='lower')
axs[1].set_title('Slope X')

axs[2].imshow(inputs[2].cpu(), cmap='RdBu', origin='lower')
axs[2].set_title('Slope Y')

axs[3].imshow(outputs[0].cpu(), cmap='Blues_r', origin='lower')
axs[3].set_title('FAT')

plt.show()


### Normalization

Since the input and output values may have very different ranges, it is important to perform normalization to both.

In [5]:
def normalize_dataset(dataset, scaler_x, scaler_y):
    min_x, max_x = scaler_x.data_min_[0], scaler_x.data_max_[0]
    min_y, max_y = scaler_y.data_min_[0], scaler_y.data_max_[0]
    normalized_dataset = []
    for idx in range(len(dataset)):
        x = dataset[idx][0]
        y = dataset[idx][1]
        norm_x = (x - min_x) / (max_x - min_x)
        norm_y = (y - min_y) / (max_y - min_y)
        normalized_dataset.append((norm_x, norm_y))
    return normalized_dataset

In [6]:
# Normalize the inputs and outputs using training dataset
scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

for idx in range(len(train_dataset)):
    scaler_x.partial_fit(train_dataset[idx][0].reshape(inputs.shape[0], -1).T.cpu())
    scaler_y.partial_fit(train_dataset[idx][1].reshape(-1, 1).cpu())

normalized_train_dataset = normalize_dataset(train_dataset, scaler_x, scaler_y)
normalized_test_dataset = normalize_dataset(test_dataset, scaler_x, scaler_y)

In [7]:
# Split dataset into train, validation, and testing
train_percnt = 0.8
train_size = int(train_percnt * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = random_split(normalized_train_dataset, [train_size, val_size])

# Model

Your main task throughout this notebook will be to define a UNET-based CNN architecture.
You are free to design the architecture as you prefer, as long as the model does its purpose and produces decent results.

In [8]:
# Create you own CNN model

# Define the model
# model = ...

# ---------------------- student exercise --------------------------------- #

# ---------------------- student exercise --------------------------------- #

# Training

Define the training and evaluation functions needed to update the model's parameters.

Tip: you can use the same training and evaluation function we have used so far.

In [9]:
def train_epoch(model, loader, optimizer, device):
    model.to(device)
    model.train() # specifies that the model is in training mode

    losses = []

    for batch in loader:
        x = batch[0].to(device)
        y = batch[1].to(device)

        # Model prediction
        preds = model(x)

        # MSE loss function
        loss = nn.MSELoss()(preds, y)

        losses.append(loss.cpu().detach())

        # Backpropagate and update weights
        loss.backward()   # compute the gradients using backpropagation
        optimizer.step()  # update the weights with the optimizer
        optimizer.zero_grad(set_to_none=True)   # reset the computed gradients

    losses = np.array(losses).mean()

    return losses

In [10]:
def evaluation(model, loader, device):
    model.to(device)
    model.eval() # specifies that the model is in evaluation mode

    losses = []

    with torch.no_grad():
        for batch in loader:
            x = batch[0].to(device)
            y = batch[1].to(device)

            # Model prediction
            preds = model(x)

            # MSE loss function
            loss = nn.MSELoss()(preds, y)
            losses.append(loss.cpu().detach())

    losses = np.array(losses).mean()

    return losses

### Define the training paramters, the optimizer, and the dataloader

In [11]:
# Set training parameters
learning_rate = 0.001
batch_size = 64
num_epochs = 100

# Create the optimizer to train the neural network via back-propagation
optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate)

# Create the training and validation dataloaders to "feed" data to the model in batches
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(normalized_test_dataset, batch_size=batch_size, shuffle=False)

## Training and validating

And finally, it's time to train you model, with many epochs.

Make sure to train for enough epochs not to end training preemptively.

Remember to save your training and validation losses to check if your model is training properly.

In [ ]:
# ---------------------- student exercise --------------------------------- #
#create vectors for the training and validation loss


# ---------------------- student exercise --------------------------------- #

In [ ]:
test_loss = evaluation(model, test_loader, device=device)
print(test_loss)

# Visualize results

## Losses

Let's check if your training and validation losses are decreasing with the epochs

In [ ]:
plt.plot(train_losses, label='Training')
plt.plot(val_losses, label='Validation')
plt.yscale('log')
plt.title('Losses')
plt.xlabel('Epochs')
plt.legend()
plt.show()

## Visualize the predictions

Select one sample from the testing dataset and apply your model.
Then, compare your prediction with the ground-truth.

In [26]:
# select one sample
data_id = 22

x = normalized_test_dataset[data_id][0].unsqueeze(0)
FAT = normalized_test_dataset[data_id][1]

# predict the FAT
pred_FAT = model(x.to(device)).detach()

Remember to denormalize your inputs and predictions before visualizing the results, so as to maintain the same real problem dimensions.

In [27]:
DEM = scaler_x.inverse_transform(x[0].reshape(3,-1).T.cpu())[:,0].reshape(64,64)
real_FAT = scaler_y.inverse_transform(FAT.reshape(-1,1).cpu()).reshape(64,64)
pred_FAT = scaler_y.inverse_transform(pred_FAT.reshape(-1,1).cpu()).reshape(64,64)

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(17,5))

diff_FAT = real_FAT - pred_FAT
max_FAT = max(pred_FAT.max(), real_FAT.max())
max_diff = max(diff_FAT.max(), -diff_FAT.min())

axs[0].imshow(DEM.squeeze(), cmap='terrain', origin='lower')
axs[1].imshow(real_FAT.squeeze(), vmin = 0, vmax=max_FAT, cmap='Blues_r', origin='lower')
axs[2].imshow(pred_FAT.squeeze(), vmin = 0, vmax=max_FAT,cmap='Blues_r', origin='lower')
axs[3].imshow(diff_FAT.squeeze(), vmin=-max_diff, vmax=max_diff, cmap='RdBu', origin='lower')

plt.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(vmin = DEM.min(), vmax=DEM.max()),
                            cmap='terrain'), fraction=0.05, shrink=0.9, ax=axs[0])
plt.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(vmin = 0, vmax=max_FAT),
                            cmap='Blues_r'), fraction=0.05, shrink=0.9, ax=axs[1])
plt.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(vmin = 0, vmax=max_FAT),
                            cmap='Blues_r'), fraction=0.05, shrink=0.9, ax=axs[2])
plt.colorbar(plt.cm.ScalarMappable(norm=TwoSlopeNorm(vmin=-max_diff, vmax=max_diff, vcenter=0),
                            cmap='RdBu'), fraction=0.05, shrink=0.9, ax=axs[3])
for ax in axs:
    ax.axis('off')

axs[0].set_title('DEM')
axs[1].set_title('Real FAT (h)')
axs[2].set_title('Predicted FAT (h)')
axs[3].set_title('Difference (h)')

plt.show()

Check the loss over the testing dataset to see which simulation is the best and which is the worst.
This will allow you to make some considerations on where your model is more suitable.

In [18]:
model.to(device)
model.eval()

all_preds = []
with torch.no_grad():
    for batch in test_loader:
        x = batch[0]
        y = batch[1]

        # Model prediction
        preds = model(x.to(device))
        all_preds.append(preds.detach().cpu())

# concatenate all predictions
all_preds = torch.cat(all_preds, dim=0)

# select all outputs from test dataset
test_FAT = torch.stack([normalized_test_dataset[i][1] for i in range(len(normalized_test_dataset))])

# loss on the test dataset per sample
test_loss = torch.stack([nn.MSELoss()(all_preds[i], test_FAT[i]) for i in range(len(all_preds))])

In [ ]:
plt.plot(test_loss.cpu())
plt.title('Test loss per sample')
plt.xlabel('Sample id')
print("The simulation with the lowest loss is the simulation number:", test_loss.argmin().item())
print("The simulation with the highest loss is the simulation number:", test_loss.argmax().item())

According to these plots, what can you derive about the model performance as a function of the topography complexity?

<!-- student -->
The model performs better on flatter topographies.
This can be motivated by simpler flow dynamics that are easier to represent with the model.
<!-- student -->